# MotoGP Riders Info - Data Preparation

This notebook cleans and standardizes the riders_info dataset based on insights from the exploration phase.

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
raw_data_path = Path("../data/raw/riders_info.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 1. Data Quality Assessment

Based on exploration findings, assess data quality issues.

In [ ]:
# Assess data quality
print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total riders: {len(df_raw)}")
print(f"\nMissing values per column:")
missing_analysis = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw)) * 100
})
print(missing_analysis)

print(f"\nData types:")
print(df_raw.dtypes)

print(f"\nBasic statistics:")
print(df_raw.describe())

## 2. Column Standardization

Standardize column names and create consistent naming.

In [ ]:
# Create working copy
df = df_raw.copy()

# Standardize column names
column_mapping = {
    'Riders All Time in All Classes': 'rider_name',
    'Victories': 'victories',
    '2nd places': 'second_places', 
    '3rd places': 'third_places',
    'Pole positions from \'74 to 2022': 'pole_positions',
    'Race fastest lap to 2022': 'fastest_laps',
    'World Championships': 'world_championships'
}

df = df.rename(columns=column_mapping)

print("Standardized columns:")
print(list(df.columns))
df.head()

## 3. Data Type Corrections

Ensure all numerical columns are properly typed.

In [ ]:
# Convert numerical columns to appropriate types
numerical_cols = ['victories', 'second_places', 'third_places', 'pole_positions', 'fastest_laps', 'world_championships']

for col in numerical_cols:
    if col in df.columns:
        # Convert to float first to handle NaN, then to int where possible
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Updated data types:")
print(df.dtypes)

print(f"\nMissing values after type conversion:")
print(df.isnull().sum())

## 4. Missing Value Treatment

Handle missing values based on data context.

In [ ]:
# Handle missing values
print("=== MISSING VALUE TREATMENT ===")

# For pole positions (data from 1974 onwards), missing likely means 0 for older riders
# But we'll keep NaN to distinguish between "no data" and "zero poles"
print(f"Pole positions missing: {df['pole_positions'].isnull().sum()} riders")

# For fastest laps, similar logic
print(f"Fastest laps missing: {df['fastest_laps'].isnull().sum()} riders")

# For podium places, missing values should be treated as 0
podium_cols = ['second_places', 'third_places']
for col in podium_cols:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        print(f"Filling {missing_count} missing values in {col} with 0")
        df[col] = df[col].fillna(0)

print(f"\nMissing values after treatment:")
print(df.isnull().sum())

## 5. Derived Metrics Creation

Create useful derived metrics for analysis.

In [ ]:
# Create derived metrics
print("=== CREATING DERIVED METRICS ===")

# Total podiums
df['total_podiums'] = df['victories'] + df['second_places'] + df['third_places']

# Podium win rate (victories / total podiums)
df['podium_win_rate'] = df['victories'] / df['total_podiums']
df['podium_win_rate'] = df['podium_win_rate'].replace([np.inf, -np.inf], np.nan)

# Championships per victory ratio
df['championships_per_victory'] = df['world_championships'] / df['victories']
df['championships_per_victory'] = df['championships_per_victory'].replace([np.inf, -np.inf], np.nan)

# Create performance categories
def categorize_performance(victories):
    if victories >= 50:
        return 'Legend'
    elif victories >= 20:
        return 'Elite'
    elif victories >= 5:
        return 'Competitive'
    elif victories >= 1:
        return 'Winner'
    else:
        return 'No Wins'

df['performance_category'] = df['victories'].apply(categorize_performance)

print("Derived metrics created:")
derived_cols = ['total_podiums', 'podium_win_rate', 'championships_per_victory', 'performance_category']
print(derived_cols)

print(f"\nPerformance category distribution:")
print(df['performance_category'].value_counts())

## 6. Rider Name Standardization

Clean and standardize rider names.

In [ ]:
# Standardize rider names
print("=== RIDER NAME STANDARDIZATION ===")

# Clean rider names
df['rider_name_clean'] = df['rider_name'].str.strip()

# Extract first and last names
df['first_name'] = df['rider_name_clean'].str.split().str[0]
df['last_name'] = df['rider_name_clean'].str.split().str[-1]

# Create display name (for visualizations)
df['display_name'] = df['last_name'] + ', ' + df['first_name']

# Check for potential duplicates
duplicate_names = df['rider_name_clean'].duplicated()
if duplicate_names.any():
    print(f"Warning: {duplicate_names.sum()} potential duplicate names found")
    print(df[duplicate_names][['rider_name_clean']].values)
else:
    print("No duplicate rider names found")

print(f"\nSample of cleaned names:")
print(df[['rider_name', 'rider_name_clean', 'display_name']].head(10))

## 7. Data Validation

Validate data consistency and logical relationships.

In [ ]:
# Data validation
print("=== DATA VALIDATION ===")

# Check that victories <= total_podiums
invalid_podiums = df[df['victories'] > df['total_podiums']]
if len(invalid_podiums) > 0:
    print(f"Warning: {len(invalid_podiums)} riders with victories > total podiums")
    print(invalid_podiums[['rider_name', 'victories', 'total_podiums']])
else:
    print("✓ All riders have victories <= total podiums")

# Check that world_championships is reasonable compared to victories
high_efficiency = df[df['championships_per_victory'] > 1]
if len(high_efficiency) > 0:
    print(f"\nNote: {len(high_efficiency)} riders with more championships than victories (very efficient!)")
    print(high_efficiency[['rider_name', 'victories', 'world_championships', 'championships_per_victory']].head())

# Check for negative values
numeric_cols = ['victories', 'second_places', 'third_places', 'world_championships']
for col in numeric_cols:
    negative_values = (df[col] < 0).sum()
    if negative_values > 0:
        print(f"Warning: {negative_values} negative values in {col}")
    else:
        print(f"✓ No negative values in {col}")

print(f"\n=== VALIDATION SUMMARY ===")
print(f"Total riders after preparation: {len(df)}")
print(f"Riders with victories: {(df['victories'] > 0).sum()}")
print(f"Riders with podiums: {(df['total_podiums'] > 0).sum()}")
print(f"Riders with championships: {(df['world_championships'] > 0).sum()}")

## 8. Final Dataset Structure

Create the final prepared dataset with consistent structure.

In [ ]:
# Define final column order
final_columns = [
    # Original data
    'rider_name', 'rider_name_clean', 'first_name', 'last_name', 'display_name',
    'victories', 'second_places', 'third_places', 
    'pole_positions', 'fastest_laps', 'world_championships',
    # Derived metrics
    'total_podiums', 'podium_win_rate', 'championships_per_victory',
    'performance_category'
]

# Create final dataset
df_final = df[final_columns].copy()

print("=== FINAL PREPARED DATASET ===")
print(f"Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\nSample of prepared data:")
print(df_final.head(10))

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nMissing values:")
print(df_final.isnull().sum())

## 9. Save Prepared Data

Save the cleaned and prepared dataset.

In [ ]:
# Create prepared data directory
prepared_data_path = Path("../../data/interim/")
prepared_data_path.mkdir(parents=True, exist_ok=True)

# Save prepared dataset
output_file = prepared_data_path / "riders_info_prepared.csv"
df_final.to_csv(output_file, index=False)

print(f"Prepared dataset saved to: {output_file}")
print(f"File size: {output_file.stat().st_size:,} bytes")

# Verification
df_verification = pd.read_csv(output_file)
print(f"\nVerification - loaded shape: {df_verification.shape}")
print("✓ File saved and verified successfully")

## 10. Preparation Summary

Summary of data preparation steps and improvements.

In [ ]:
print("=== PREPARATION SUMMARY ===")
print("\n✅ COMPLETED TASKS:")
print("1. Column name standardization")
print("2. Data type corrections and validation")
print("3. Missing value treatment (podium places filled with 0)")
print("4. Derived metrics creation (total_podiums, win_rates, etc.)")
print("5. Rider name cleaning and standardization")
print("6. Performance categorization")
print("7. Data validation and consistency checks")
print("8. Prepared dataset saved for analysis")

print("\n📊 DATASET IMPROVEMENTS:")
print(f"- Standardized column names for consistency")
print(f"- Added {len([col for col in df_final.columns if col not in df_raw.columns])} derived metrics")
print(f"- Performance categories for easy analysis")
print(f"- Cleaned rider names with multiple formats")
print(f"- Validated data relationships")

print("\n🔍 DATA QUALITY NOTES:")
print(f"- Pole positions data available from 1974 onwards only")
print(f"- Some fastest lap data missing for certain riders")
print(f"- All logical consistency checks passed")
print(f"- No duplicate rider names found")

print("\n➡️  READY FOR ANALYSIS PHASE")
print("The prepared dataset is now ready for specific business question analysis.")